In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df=pd.read_csv('ecommerce_customer_churn_messy.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [24]:
df.columns

Index(['customer_id', 'signup_date', 'gender', 'customer_age', 'country',
       'city', 'customer_segment', 'membership_type', 'acquisition_channel',
       'payment_method', 'preferred_device', 'product_category',
       'subscription_type', 'subscription_start', 'subscription_end',
       'annual_income', 'account_balance', 'total_spending',
       'average_order_value', 'number_of_orders', 'days_since_last_order',
       'last_purchase_date', 'last_login', 'website_visits', 'support_tickets',
       'discount_usage', 'refund_amount', 'lifetime_value',
       'cancellation_reason', 'cancellation_date', 'final_account_status',
       'churn'],
      dtype='str')

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
df.describe(include="all")

In [ ]:
df.isnull().sum()

In [ ]:
df.isnull().mean() * 100

In [ ]:
df.duplicated().sum()

In [ ]:
df["churn"].value_counts()

In [ ]:
df["churn"].value_counts(normalize=True) * 100

In [18]:
df = df.drop_duplicates()

In [19]:
df.shape

(21712, 32)

In [22]:
missing_tokens = [
    "",
    "na",
    "n/a",
    "nan",
    "null",
    "none",
    "unknown",
    "?"
]

In [23]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].replace(
        missing_tokens,
        np.nan
    )

C:\Users\User\AppData\Local\Temp\ipykernel_8276\1610229325.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [26]:
gender_map = {
    "female": "Female",
    "f": "Female",
    "male": "Male",
    "m": "Male"
}

df["gender"] = (
    df["gender"]
    .str.strip()
    .str.lower()
    .map(gender_map)
)

In [27]:
df["gender"].value_counts()

gender
Female    9874
Male      9741
Name: count, dtype: int64

In [29]:
df["membership_type"] = (
    df["membership_type"]
    .str.strip()
    .str.lower()
    .replace({
        "basic": "Basic",
        "premium": "Premium",
        "gold": "Gold",
        "platinum": "Platinum"
    })
)

In [30]:
df["membership_type"].value_counts()

membership_type
Basic       8094
Premium     6886
Gold        4344
Platinum    1300
-            128
Name: count, dtype: int64

In [33]:
def parse_number(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()

    if value in ["", "na", "n/a", "unknown", "?", "-", "null"]:
        return np.nan

    value = value.replace(",", "")
    value = value.replace("$", "")
    value = value.replace("usd", "")

    if value.endswith("k"):
        return float(value[:-1]) * 1000

    return float(value)

In [31]:
numeric_columns = [
    "customer_age",
    "annual_income",
    "account_balance",
    "total_spending",
    "average_order_value",
    "number_of_orders",
    "days_since_last_order",
    "website_visits",
    "support_tickets",
    "discount_usage",
    "refund_amount",
    "lifetime_value"
]

In [34]:
for col in numeric_columns:
    df[col] = df[col].apply(parse_number)

In [35]:
df[numeric_columns].dtypes

customer_age             float64
annual_income            float64
account_balance          float64
total_spending           float64
average_order_value      float64
number_of_orders         float64
days_since_last_order    float64
website_visits           float64
support_tickets          float64
discount_usage           float64
refund_amount            float64
lifetime_value           float64
dtype: object

In [36]:
df.loc[
    ~df["customer_age"].between(18, 100),
    "customer_age"
] = np.nan

In [37]:
df.loc[
    ~df["customer_age"].between(18, 100),
    "customer_age"
] = np.nan

In [38]:
non_negative_cols = [
    "annual_income",
    "total_spending",
    "average_order_value",
    "number_of_orders",
    "days_since_last_order",
    "website_visits",
    "support_tickets",
    "discount_usage",
    "refund_amount",
    "lifetime_value"
]

for col in non_negative_cols:
    df.loc[df[col] < 0, col] = np.nan

In [40]:
df["last_login"] = pd.to_datetime(
    df["last_login"],
    format="mixed",
    errors="coerce"
)

In [41]:
df["last_login_year"] = df["last_login"].dt.year
df["last_login_month"] = df["last_login"].dt.month
df["last_login_dayofweek"] = df["last_login"].dt.dayofweek

In [43]:
reference_date = pd.Timestamp("2025-07-01")

df["days_since_login"] = (
    reference_date - df["last_login"]
).dt.days

In [44]:
df["days_since_login"]

0        137.0
1          6.0
2          NaN
3          5.0
4         10.0
         ...  
22074      9.0
22076     14.0
22077     18.0
22078     13.0
22079      NaN
Name: days_since_login, Length: 21712, dtype: float64

In [45]:
X = df.drop(columns="churn")
y = df["churn"]

In [48]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)